<a href="https://colab.research.google.com/github/riofutabac/PlacasVideos/blob/main/GoogleColab_ALPR_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Pipeline ALPR Orientado a Eventos — Vía de Lastre (Píntag)

**Arquitectura:** Crop físico + Motion Gate de 3 estados + YOLOv8 + ByteTrack + Top-M/Top-K + FastPlateOCR + Deduplicación + Reporte Excel con fotos incrustadas.

Repositorio: [`riofutabac/PlacasVideos`](https://github.com/riofutabac/PlacasVideos) (rama `main`).

> **Entorno de ejecución:** antes de correr cualquier celda, ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución` y selecciona **GPU (T4)**. Sin GPU, el pipeline funcionará pero mucho más lento.

## 1. Verificar GPU disponible

In [1]:
# Comprobar que Colab asignó una GPU NVIDIA (T4) y que PyTorch la detecta
!nvidia-smi

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).")


Tue Sep 22 15:15:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar Google Drive

In [2]:
# Los videos de la cámara viven en Google Drive, dentro de la carpeta "Cam PL"
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 3. Clonar o actualizar el repositorio

In [3]:
import os

REPO_URL = "https://github.com/riofutabac/PlacasVideos.git"
REPO_DIR = "/content/PlacasVideos"
BRANCH = "main"

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("El repositorio ya existe, actualizando con git pull...")
    !cd "$REPO_DIR" && git fetch origin && git checkout "$BRANCH" && git pull origin "$BRANCH"

%cd $REPO_DIR
!git status


Cloning into '/content/PlacasVideos'...
remote: Enumerating objects: 590, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 590 (delta 116), reused 122 (delta 59), pack-reused 388 (from 1)
Receiving objects: 100% (590/590), 14.55 MiB | 32.75 MiB/s, done.
Resolving deltas: 100% (357/357), done.
/content/PlacasVideos
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 4. Instalar dependencias

Instalamos las dependencias del `requirements.txt` más las que Colab necesita de forma
específica: `onnxruntime-gpu` (compatible con CUDA 12, en vez del `onnxruntime` de CPU),
`fast-alpr`, y `onnx` (lo pide `ultralytics` para exportar, y si no está preinstalado
pide reiniciar el entorno de ejecución a mitad de la corrida).

In [4]:
# Instalar dependencias base del proyecto
!pip install -q -r requirements.txt
!pip install -q fast-alpr onnx

# Evitar conflicto de CUDA 13 en Colab: usar onnxruntime-gpu compatible con CUDA 12
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.22.0

# Fijar versión de supervision compatible con el pipeline
!pip install -q "supervision<0.31" "opencv-python-headless<5"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.6/391.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 34.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [5]:
# Verificar versiones instaladas de las librerías clave
import cv2
import onnxruntime as ort
import supervision as sv
import ultralytics

print(f"OpenCV: {cv2.__version__}")
print(f"ONNX Runtime: {ort.__version__} | providers: {ort.get_available_providers()}")
print(f"Supervision: {sv.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
OpenCV: 4.14.0
ONNX Runtime: 1.22.0 | providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Supervision: 0.30.5
Ultralytics: 8.4.159


## 5. Configuración

In [6]:
# Carpeta de Drive donde están los videos de la cámara
VIDEOS_DIR = "/content/drive/MyDrive/Cam PL"

# Clips específicos para la corrida de validación rápida (60 y 61 tienen ground truth)
CLIPS = [60, 61]

# Límite de videos para una corrida de producción parcial (None = sin límite)
LIMIT = None


## 6. Corrida de validación rápida (clips 60 y 61)

Estos dos clips tienen datos de referencia (*ground truth*). Al incluirlos, `main.py`
imprime automáticamente al final un bloque **GROUND TRUTH EVALUATION** con precisión,
recall y las placas que coinciden o no contra el ground truth — úsalo para confirmar
que el pipeline sigue funcionando correctamente después de cualquier cambio.

In [7]:
!python main.py "$VIDEOS_DIR" --clips {" ".join(map(str, CLIPS))}


PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrida: 2026-09-21 21:38:26
📹 Videos a procesar (2 archivos):
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(61).mp4
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading FastALPR on providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] (device='cuda')...
INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
⚡ [SSD Staging] Copiando Camara Placas 2_20260

## 7. Corrida de producción

Procesa toda la carpeta de videos, o usa `--limit` para procesar solo los primeros N
archivos (útil para pruebas intermedias antes de lanzar la carpeta completa).

In [7]:
!git log --oneline -1

9b77244 (HEAD -> main, origin/main, origin/HEAD) docs: add precision-phase closeout report (137 vs 133 run comparison)


In [9]:
# Toda la carpeta:
!python main.py "$VIDEOS_DIR"

# O bien, limitar la cantidad de videos a procesar (descomenta y ajusta LIMIT en la celda de configuración):
# !python main.py "$VIDEOS_DIR" --limit 10


PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrida: 2026-09-21 21:40:32
📹 Videos a procesar (63 archivos):
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(1).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(2).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(3).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(4).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(5).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(6).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(7).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(8).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(9).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(10).mp4
  - /co

## 8. Descargar el reporte Excel de auditoría

In [10]:
import os
from google.colab import files

report_path = "reports/reporte_auditoria.xlsx"
if os.path.exists(report_path):
    files.download(report_path)
    print(f"📥 Descargando {report_path}...")
else:
    print(f"⚠️ No se encontró {report_path}. Corre el pipeline primero (secciones 6 o 7).")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando reports/reporte_auditoria.xlsx...


In [11]:
!git pull
!python benchmarks/build_gt_candidates.py


Already up to date.
📋 [GT Builder] Leídas 39 placas auditadas del Excel.

📊 === Resumen de Cruce con Base de Datos (data/events.sqlite) ===
  • Tolerancia temporal: 75s
  • Total vehículos auditados en Excel: 39
  • Total eventos únicos en el sistema: 137
  • Coincidencia Exacta de Placa: 17 (43.6%)
  • Coincidencia Cercana (1-2 caracteres, mismo vehículo con error OCR): 3 (7.7%)
  • Solo Vehículo (detectado, placa no leída): 15 (38.5%)
  • No encontrados / Sin cruce: 4 (10.3%)
  • Eventos adicionales detectados (no listados en auditoría; esperado, la auditoría solo registra vehículos que cruzaron ambas cámaras): 102

#    Placa GT  Tipo    Hora Cam2  Tipo Match       Placa Pipeline  Diff (s)  Dist  Conf OCR
------------------------------------------------------------------------------------------------
001  TCX0795   truck   13:13:00   NEAR_PLATE       TRX0495         32.0s     2     0.87    
002  PBO4275   truck   11:07:00   EXACT_PLATE      PBO4275         34.0s     0     1.00    
0

## 9. Benchmarks / Diagnóstico (opcional)

Las siguientes celdas **no son necesarias** para generar el reporte de auditoría.
Sirven para diagnosticar rendimiento y comparar backends de inferencia/decodificación.
Ejecuta solo la(s) que necesites.

### 9.1 Benchmark de runtime del detector de vehículos (requiere GPU)

In [12]:
# Compara ultralytics vs ONNX Runtime (I/O binding) vs TensorRT para el modelo YOLO
!python benchmarks/benchmark_vehicle_runtime.py --model yolov8n.onnx


BENCHMARK DE RUNTIMES DE VEHÍCULO YOLOv8n (Fase C2/C3)
🖼️ Evaluando 20 cuadros ROI (2300x1064) @ imgsz=416...
⏱️ Probando backend: Ultralytics YOLO (Default)...
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
⏱️ Probando backend: ONNX Runtime CUDA I/O Binding...
⏱️ Probando backend: ONNX Runtime TensorRT EP (FP16)...
2026-09-21 22:33:07.985323819 [E:onnxruntime:Default, provider_bridge_ort.cc:2167 TryGetProviderInfo_TensorRT] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1778 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_pro

### 9.2 Diagnóstico A/B de decodificadores de video (NVDEC vs OpenCV)

In [13]:
# Compara el decodificador NVDEC (GPU) contra OpenCV (CPU) sobre el pipeline completo
!python benchmarks/diagnose_ab_decoders.py --full-pipeline


🔍 Paso 1/2: Diagnóstico numérico exacto en timestamps críticos (25, 105, 109, 112, 116, 120s)...
DIAGNÓSTICO COMPARATIVO A/B EXACTO: OpenCV vs NVDEC
Video: /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4

Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading FastALPR on providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] (device='cuda')...
INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
[OPENCV] Extrayendo frames en timestamps [25.0, 105.0, 109.0, 112.0, 116.0, 120.0]...
ℹ️ [VideoDecoder

### 9.3 Benchmark puro de FFmpeg + NVDEC

In [14]:
!python benchmarks/benchmark_ffmpeg_nvdec.py


BENCHMARK DE PIPELINES FFMPEG NVDEC (Clip 60: 8223 frames, 330.5s)
Video: /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4

Testing: 1. NVDEC CUDA -> hwdownload -> NV12 (null sink)...
Traceback (most recent call last):
  File "/usr/lib/python3.13/subprocess.py", line 2154, in _communicate
    ready = selector.select(timeout)
  File "/usr/lib/python3.13/selectors.py", line 398, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/PlacasVideos/benchmarks/benchmark_ffmpeg_nvdec.py", line 133, in <module>
    main()
    ~~~~^^
  File "/content/PlacasVideos/benchmarks/benchmark_ffmpeg_nvdec.py", line 50, in main
    run_bench("1. NVDEC CUDA -> hwdownload -> NV12 (null sink)", [
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        "ffmpeg", "-hide_banner", "-nostdin", "-loglevel", "error",
  

### 9.4 Resumen ejecutivo de la última corrida

In [15]:
# Resumen con métricas clave y costo estimado de GPU en la nube
!python benchmarks/resumen_ejecutivo.py --cost-per-hour 0.35



╔══════════════════════════════════════════════════════════════════════╗
║           RESUMEN EJECUTIVO DE RENDIMIENTO - PIPELINE ALPR           ║
╚══════════════════════════════════════════════════════════════════════╝

📌 Identificador de Corrida:  RUN_20260921_214037
🕒 Inicio: 2026-09-21 21:40:37  |  Fin: 2026-09-21 22:32:54
📁 Clips de video procesados: 61 archivos (resolución 3K: 2960×1664)

────────────────────────────────────────────────────────────────────────
⏱️  VELOCIDAD Y TIEMPOS DE PROCESAMIENTO
────────────────────────────────────────────────────────────────────────
 • Metraje de video analizado:      320.6 minutos  (19,233.3 segundos)
 • Tiempo real de reloj tomado:     44.06 minutos  (2,643.6 segundos)
 • Tasa de Aceleración:              7.28× TIEMPO REAL
 • Velocidad de cómputo:            181.6 frames/segundo equivalentes
   ➔ Conclusión: El sistema procesa 1 hora de video en apenas 8.2 minutos.

────────────────────────────────────────────────────────────────────────


## 10. Solución de problemas

- **"archivo dañado" / error al abrir un clip:** revisa el error real impreso *justo
  arriba* de ese mensaje en la salida de la celda — normalmente indica la causa
  concreta (códec no soportado, archivo incompleto en Drive, etc.), y el mensaje de
  "archivo dañado" es solo el resumen final.
- **Cambiaste dependencias (paso 4) y algo falla de forma rara:** reinicia el entorno
  de ejecución (`Entorno de ejecución → Reiniciar entorno de ejecución`) y vuelve a
  ejecutar desde el paso 2 en adelante. Reinstalar `onnxruntime-gpu` u `onnx` sin
  reiniciar puede dejar el proceso de Python con la versión vieja cargada en memoria.
- **`git pull` falla con conflictos:** el repo en `/content/PlacasVideos` puede tener
  cambios locales de una corrida anterior. Bórralo (`!rm -rf /content/PlacasVideos`) y
  vuelve a ejecutar el paso 3 para clonar limpio.
- **No aparece el bloque GROUND TRUTH EVALUATION:** solo se imprime cuando la corrida
  incluye los clips 60 y/o 61 (`--clips 60 61`), porque son los únicos con datos de
  referencia.

In [16]:
!git pull

Already up to date.


In [17]:
!python tools/inspect_missing_clips.py --video-dir "/content/drive/MyDrive/Cam PL" --output-dir "reports/video_forensics"

🎬 [Forense] Procesando PAZ1513 (Camión JAC plataforma) en Camara Placas 2_20260909105651-20260909163038(31).mp4 (FPS: 25.0, Duración: 329.3s)
  ✅ Guardada hoja de contacto con 66 cuadros en: reports/video_forensics/forense_PAZ1513_clip31.jpg
🎬 [Forense] Procesando PAB7630 (Mixer de concreto tambor) en Camara Placas 2_20260909105651-20260909163038(37).mp4 (FPS: 25.0, Duración: 328.2s)
  ✅ Guardada hoja de contacto con 41 cuadros en: reports/video_forensics/forense_PAB7630_clip37.jpg
🎬 [Forense] Procesando PAB3439 (Camión furgón blanco) en Camara Placas 2_20260909105651-20260909163038(57).mp4 (FPS: 24.9, Duración: 330.4s)
  ✅ Guardada hoja de contacto con 41 cuadros en: reports/video_forensics/forense_PAB3439_clip57.jpg


In [18]:
!git pull origin main
!python main.py "$VIDEOS_DIR" --clips 37 57

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 13.32 KiB | 1.90 MiB/s, done.
From https://github.com/riofutabac/PlacasVideos
 * branch            main       -> FETCH_HEAD
   72a3d98..7693007  main       -> origin/main
Updating 72a3d98..7693007
Fast-forward
 GoogleColab_ALPR_Pipeline.ipynb | 1849 ++++++++++++++++++---------------------
 1 file changed, 842 insertions(+), 1007 deletions(-)
PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrida: 2026-09-21 22:40:38
📹 Videos a procesar (2 archivos):
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(37).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(57).mp4
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for 

In [19]:
!python benchmarks/build_gt_candidates.py

📋 [GT Builder] Leídas 39 placas auditadas del Excel.

📊 === Resumen de Cruce con Base de Datos (data/events.sqlite) ===
  • Tolerancia temporal: 75s
  • Total vehículos auditados en Excel: 39
  • Total eventos únicos en el sistema: 137
  • Coincidencia Exacta de Placa: 17 (43.6%)
  • Coincidencia Cercana (1-2 caracteres, mismo vehículo con error OCR): 3 (7.7%)
  • Solo Vehículo (detectado, placa no leída): 15 (38.5%)
  • No encontrados / Sin cruce: 4 (10.3%)
  • Eventos adicionales detectados (no listados en auditoría; esperado, la auditoría solo registra vehículos que cruzaron ambas cámaras): 102

#    Placa GT  Tipo    Hora Cam2  Tipo Match       Placa Pipeline  Diff (s)  Dist  Conf OCR
------------------------------------------------------------------------------------------------
001  TCX0795   truck   13:13:00   NEAR_PLATE       TRX0495         32.0s     2     0.87    
002  PBO4275   truck   11:07:00   EXACT_PLATE      PBO4275         34.0s     0     1.00    
003  PFM8305   truck 

In [20]:
from google.colab import files
files.download('data/events.sqlite')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!git pull

remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 19 (delta 11), reused 18 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (19/19), 14.56 KiB | 1.82 MiB/s, done.
From https://github.com/riofutabac/PlacasVideos
   9b77244..2643b63  main       -> origin/main
Updating 9b77244..2643b63
Fast-forward
 benchmarks/diagnose_plate_loss.py | 103 ++++++++++++++++++++++
 config/camera_config.yaml         |   3 +
 main.py                           |  81 +++++++++++++-----
 src/db_manager.py                 |  65 ++++++++++++--
 src/pipeline_runner.py            | 123 ++++++++++++++-------------
 src/plate_processor.py            | 174 ++++----------------------------------
 src/plate_voting.py               | 162 +++++++++++++++++++++++++++++++++++
 src/rejection_logger.py           |  70 +++++++++++++++
 src/tracking_manager.py           |  16 +++-
 src/vehicle_filtering.py          |  76 ++++

In [11]:
!python main.py "$VIDEOS_DIR" --diagnose

PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrida: 2026-09-22 15:27:30
📹 Videos a procesar (63 archivos):
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(1).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(2).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(3).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(4).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(5).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(6).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(7).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(8).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(9).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(10).mp4
  - /co

In [13]:
from google.colab import files
files.download('reports/diagnostics/RUN_20260922_152743.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!git pull
!python main.py "$VIDEOS_DIR" --prefetch

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 11 (delta 6), reused 11 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 7.94 KiB | 1.59 MiB/s, done.
From https://github.com/riofutabac/PlacasVideos
   2643b63..6f63b88  main       -> origin/main
Updating 2643b63..6f63b88
Fast-forward
 benchmarks/compare_runs.py    | 142 ++++++++++++++++++++++++++++++++++++++++
 config/camera_config.yaml     |   1 +
 main.py                       | 148 ++++++++++++++++++++++++++++--------------
 src/clip_prefetcher.py        | 105 ++++++++++++++++++++++++++++++
 tests/test_clip_prefetcher.py | 142 ++++++++++++++++++++++++++++++++++++++++
 5 files changed, 490 insertions(+), 48 deletions(-)
 create mode 100644 benchmarks/compare_runs.py
 create mode 100644 src/clip_prefetcher.py
 create mode 100644 tests/test_clip_prefetcher.py
PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrid